Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo\\SNpollutionFinal.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [5]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [6]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [7]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [8]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [9]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [10]:
datosNormalizados.shape

(43800, 6)

In [11]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


Espacio de búsqueda

In [12]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [13]:
futuros = 24
pasados  = 12

In [14]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 0])


In [15]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43765, 12, 6)
Dimensiones de Y: (43765, 1)


In [16]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894 ]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347 ]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449]
 [ 0.57004095 -0.65625711 -1.34931411  0.90832835 -0.38094383 -0.09555657]]


Se dividen nuevamente los conjuntos de datos

In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30635, 12, 6)
Las dimensiones de testX son:  (8797, 12, 6)
Las dimensiones de valX son:  (4333, 12, 6)


In [18]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30635, 1)
Las dimensiones de testY son:  (8797, 1)
Las dimensiones de valY son:  (4333, 1)


Se crean métricas para medir desempeño

In [19]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [20]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [21]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=128,
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [22]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

240/240 - 16s - 67ms/step - ia: 0.1933 - loss: 1.1486 - mae: 0.8573 - rmse: 1.0668 - smape: 1.6114 - val_ia: 0.2558 - val_loss: 0.9829 - val_mae: 0.7633 - val_rmse: 0.9006 - val_smape: 1.7882

Epoch 2/128                                           

240/240 - 4s - 17ms/step - ia: 0.1688 - loss: 1.0454 - mae: 0.7746 - rmse: 1.0158 - smape: 1.6228 - val_ia: 0.2529 - val_loss: 0.9678 - val_mae: 0.7367 - val_rmse: 0.8786 - val_smape: 1.9922

Epoch 3/128                                           

240/240 - 5s - 20ms/step - ia: 0.1583 - loss: 1.0316 - mae: 0.7619 - rmse: 1.0100 - smape: 1.6291 - val_ia: 0.2529 - val_loss: 0.9664 - val_mae: 0.7337 - val_rmse: 0.8763 - val_smape: 1.9217

Epoch 4/128                                           

240/240 - 3s - 13ms/step - ia: 0.1448 - loss: 1.0218 - mae: 0.7577 - rmse: 1.0047 - smape: 1.6595 - val_ia: 0.2529 - val_loss: 0.9653 - val_mae: 0.7325 - val_rmse: 0.8751 - val_smape: 1.8984

Epoch 5

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                        

1915/1915 - 66s - 34ms/step - ia: 0.3785 - loss: 0.8236 - mae: 0.6675 - rmse: 0.8741 - smape: 1.2993 - val_ia: 0.2593 - val_loss: 0.7889 - val_mae: 0.6318 - val_rmse: 0.7091 - val_smape: 1.1579

Epoch 2/128                                                                        

1915/1915 - 44s - 23ms/step - ia: 0.4100 - loss: 0.7880 - mae: 0.6482 - rmse: 0.8533 - smape: 1.2458 - val_ia: 0.2576 - val_loss: 0.7738 - val_mae: 0.6294 - val_rmse: 0.7061 - val_smape: 1.1619

Epoch 3/128                                                                        

1915/1915 - 49s - 26ms/step - ia: 0.4205 - loss: 0.7711 - mae: 0.6395 - rmse: 0.8451 - smape: 1.2303 - val_ia: 0.2490 - val_loss: 0.7476 - val_mae: 0.6456 - val_rmse: 0.7116 - val_smape: 1.2913

Epoch 4/128                                                                        

1915/1915 - 48s - 25ms/step - ia: 0.4288 - loss: 0.7588 - mae: 0.6344 - rms

...

Epoch 1/128                                                                        

1915/1915 - 38s - 20ms/step - ia: 0.2838 - loss: 1.0916 - mae: 0.7850 - rmse: 1.0075 - smape: 1.4587 - val_ia: 0.2238 - val_loss: 0.8466 - val_mae: 0.6956 - val_rmse: 0.7546 - val_smape: 1.5568

Epoch 2/128                                                                        

1915/1915 - 24s - 13ms/step - ia: 0.3345 - loss: 0.9029 - mae: 0.7081 - rmse: 0.9174 - smape: 1.3730 - val_ia: 0.2404 - val_loss: 0.7842 - val_mae: 0.6588 - val_rmse: 0.7236 - val_smape: 1.3382

Epoch 3/128                                                                        

1915/1915 - 24s - 12ms/step - ia: 0.3620 - loss: 0.8604 - mae: 0.6870 - rmse: 0.8963 - smape: 1.3222 - val_ia: 0.2399 - val_loss: 0.7815 - val_mae: 0.6618 - val_rmse: 0.7271 - val_smape: 1.3362

Epoch 4/128                                                                        

1915/1915 - 41s - 21ms/step - ia: 0.3637 - loss: 0.8529 - mae: 0.6837 - rms

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                          

3830/3830 - 39s - 10ms/step - ia: 0.2461 - loss: 1.0636 - mae: 0.7918 - rmse: 0.9714 - smape: 1.6345 - val_ia: 0.1797 - val_loss: 1.0009 - val_mae: 0.7695 - val_rmse: 0.8065 - val_smape: 1.7668

Epoch 2/128                                                                          

3830/3830 - 30s - 8ms/step - ia: 0.2497 - loss: 1.0568 - mae: 0.7872 - rmse: 0.9671 - smape: 1.6329 - val_ia: 0.1805 - val_loss: 0.9945 - val_mae: 0.7646 - val_rmse: 0.8015 - val_smape: 1.7692

Epoch 3/128                                                                          

3830/3830 - 25s - 7ms/step - ia: 0.2474 - loss: 1.0432 - mae: 0.7816 - rmse: 0.9617 - smape: 1.6348 - val_ia: 0.1809 - val_loss: 0.9890 - val_mae: 0.7604 - val_rmse: 0.7973 - val_smape: 1.7709

Epoch 4/128                                                                          

3830/3830 - 28s - 7ms/step - ia: 0.2463 - loss: 1.0413 - mae: 0.7799 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                          

240/240 - 9s - 36ms/step - ia: 0.2966 - loss: 0.9969 - mae: 0.7479 - rmse: 0.9913 - smape: 1.4226 - val_ia: 0.2988 - val_loss: 0.7960 - val_mae: 0.6692 - val_rmse: 0.8077 - val_smape: 1.4012

Epoch 2/128                                                                          

240/240 - 4s - 16ms/step - ia: 0.3620 - loss: 0.8968 - mae: 0.7057 - rmse: 0.9413 - smape: 1.3279 - val_ia: 0.3157 - val_loss: 0.7703 - val_mae: 0.6431 - val_rmse: 0.7887 - val_smape: 1.2741

Epoch 3/128                                                                          

240/240 - 4s - 16ms/step - ia: 0.3748 - loss: 0.8716 - mae: 0.6935 - rmse: 0.9285 - smape: 1.3057 - val_ia: 0.3167 - val_loss: 0.7700 - val_mae: 0.6446 - val_rmse: 0.7901 - val_smape: 1.2794

Epoch 4/128                                                                          

240/240 - 4s - 16ms/step - ia: 0.3766 - loss: 0.8559 - mae: 0.6859 - rmse: 0

In [23]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}
